# Process Library Runner

This notebook generalizes the existing methanol workflow into a YAML-driven process library. It discovers process folders automatically, validates each `process.yaml`, performs a pre-run Codex-style coherence review, proposes YAML improvements when the coherence gate fails, and only executes processes whose specifications pass that gate.

## 1. Environment setup and imports

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd()).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from aspen_automation import (
    analyze_process_spec_coherence,
    apply_process_spec_improvements,
    build_codex_improvement_markdown,
    build_codex_results_markdown,
    build_codex_spec_markdown,
    load_process_spec,
    load_result_artifact_tables,
    run_process,
    scan_process_library,
    suggest_process_spec_improvements,
    validate_process_spec_file,
    write_process_spec_file,
)


## 1.5 Aspen Plus pre-flight check

In [ ]:
from aspen_automation import check_aspen_running, AspenNotRunningError

if not check_aspen_running():
    raise AspenNotRunningError()

print('Aspen Plus is running. Ready to proceed.')

## 2. Repository path resolution

In [ ]:
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
PROCESS_LIBRARY_DIR = REPO_ROOT / "process_library"
PROCESS_RUNS_DIR = REPO_ROOT / "process_runs"

print(f"Repository root: {REPO_ROOT}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Process library: {PROCESS_LIBRARY_DIR}")
print(f"Process runs: {PROCESS_RUNS_DIR}")


## 3. Process library path configuration

In [ ]:
LIBRARY_ROOT = PROCESS_LIBRARY_DIR
RUNS_ROOT = PROCESS_RUNS_DIR
VISIBLE = False
BUILD_MODE = "auto"
TIMEOUT_SECONDS = 1800
BUILD_TIMEOUT_SECONDS = 180
REPORT_FORMAT = "html"
ONLY_PROCESSES: set[str] | None = None

print("Notebook configuration")
print(f"- LIBRARY_ROOT={LIBRARY_ROOT}")
print(f"- RUNS_ROOT={RUNS_ROOT}")
print(f"- VISIBLE={VISIBLE}")
print(f"- BUILD_MODE={BUILD_MODE}")
print("  auto = COM block-by-block builder, com-auto = legacy INP/COM import diagnostics")
print(f"- TIMEOUT_SECONDS={TIMEOUT_SECONDS}")
print(f"- BUILD_TIMEOUT_SECONDS={BUILD_TIMEOUT_SECONDS}")
print(f"- REPORT_FORMAT={REPORT_FORMAT}")
print(f"- ONLY_PROCESSES={ONLY_PROCESSES}")


## 4. YAML schema / expected fields overview

Each `process.yaml` uses the existing plant specification schema already supported by `aspen_automation`.

Required top-level sections:

- `metadata`
- `components`
- `properties`
- `flowsheet`
- `streams`
- `blocks`

Optional sections already supported by the generator:

- `flowsheeting_options`
- `chemistry`
- `reaction_sets`
- `targets`


## 5. Discovery of all available process folders

In [ ]:
scan = scan_process_library(LIBRARY_ROOT)

selected_processes = [
    process
    for process in scan.processes
    if ONLY_PROCESSES is None or process.name in ONLY_PROCESSES
]

process_rows = [
    {
        "process_name": process.name,
        "process_dir": str(process.process_dir),
        "spec_path": str(process.spec_path),
    }
    for process in selected_processes
]
issue_rows = [
    {
        "process_name": issue.process_name,
        "process_dir": str(issue.process_dir),
        "message": issue.message,
    }
    for issue in scan.issues
]

print(f"Discovered {len(process_rows)} runnable process(es) after filtering.")
display(pd.DataFrame(process_rows))

if issue_rows:
    print(f"Found {len(issue_rows)} discovery issue(s).")
    display(pd.DataFrame(issue_rows))


## 6. Shared helper functions

In [ ]:
def summarize_process_result(result) -> dict[str, str]:
    acceptance = result.details.get("acceptance") if result.details else None
    acceptance_value = ""
    if isinstance(acceptance, dict) and "passed" in acceptance:
        acceptance_value = str(acceptance["passed"])

    generated_files = ", ".join(result.generated_files[:5])
    if len(result.generated_files) > 5:
        generated_files += ", ..."

    return {
        "process_name": result.process_name,
        "status": result.status,
        "acceptance_passed": acceptance_value,
        "report_dir": str(result.report_dir) if result.report_dir else "",
        "generated_files": generated_files,
        "error": result.error or "",
    }


def display_process_banner(name: str) -> None:
    display(Markdown(f"## Process: `{name}`"))


def first_issue_message(report: dict | None, *, severity: str = "error") -> str:
    if not isinstance(report, dict):
        return ""
    for issue in report.get("issues", report.get("errors", [])):
        if issue.get("severity") == severity:
            return str(issue.get("message", ""))
    return ""


def parse_improvement_selection(response: str, suggestions: list[dict]) -> list[str]:
    auto_suggestions = [suggestion for suggestion in suggestions if suggestion.get("auto_applicable")]
    normalized = response.strip().lower()
    if normalized in {"", "n", "no"}:
        return []
    if normalized in {"y", "yes", "all"}:
        return [suggestion["id"] for suggestion in auto_suggestions]

    selected_ids: list[str] = []
    for token in normalized.split(","):
        token = token.strip()
        if not token.isdigit():
            continue
        index = int(token) - 1
        if 0 <= index < len(auto_suggestions):
            selected_ids.append(auto_suggestions[index]["id"])
    return selected_ids


## 7. YAML coherence analysis per discovered process

In [ ]:
validation_reports: dict[str, dict] = {}
coherence_reports: dict[str, dict] = {}
process_specs: dict[str, dict] = {}
coherence_rows: list[dict[str, str]] = []

for process in selected_processes:
    display_process_banner(process.name)
    print(f"Process directory: {process.process_dir}")
    print(f"Spec path: {process.spec_path}")

    validation_report = validate_process_spec_file(process.spec_path)
    validation_reports[process.name] = validation_report

    if not validation_report.get("valid", False):
        display(Markdown(build_codex_spec_markdown(process.name, process.spec_path, validation_report, None)))
        coherence_rows.append(
            {
                "process_name": process.name,
                "validation_passed": "False",
                "coherence_passed": "False",
                "blocking_issue": first_issue_message({"issues": validation_report.get("errors", [])}) or "Validation failed",
            }
        )
        continue

    spec = load_process_spec(process.process_dir)
    process_specs[process.name] = spec
    coherence_report = analyze_process_spec_coherence(spec)
    coherence_reports[process.name] = coherence_report

    display(
        Markdown(
            build_codex_spec_markdown(
                process.name,
                process.spec_path,
                validation_report,
                coherence_report,
                spec=spec,
            )
        )
    )

    coherence_rows.append(
        {
            "process_name": process.name,
            "validation_passed": "True",
            "coherence_passed": str(coherence_report.get("passed", False)),
            "blocking_issue": first_issue_message(coherence_report) or "",
        }
    )

if coherence_rows:
    display(pd.DataFrame(coherence_rows))


## 8. Suggested YAML improvements per discovered process

In [ ]:
improvement_plans: dict[str, list[dict]] = {}

for process in selected_processes:
    display_process_banner(process.name)

    validation_report = validation_reports.get(process.name)
    coherence_report = coherence_reports.get(process.name)
    spec = process_specs.get(process.name)

    if not isinstance(validation_report, dict) or not validation_report.get("valid", False):
        print("Skipping YAML improvement suggestions because structural validation did not pass.")
        continue

    if spec is None:
        spec = load_process_spec(process.process_dir)
        process_specs[process.name] = spec

    suggestions = suggest_process_spec_improvements(spec, coherence_report)
    improvement_plans[process.name] = suggestions
    display(Markdown(build_codex_improvement_markdown(process.name, suggestions)))

    auto_suggestions = [suggestion for suggestion in suggestions if suggestion.get("auto_applicable")]
    if not isinstance(coherence_report, dict) or coherence_report.get("passed", False):
        print("No blocking coherence issues remain, so no YAML edits are required before execution.")
        continue

    if not auto_suggestions:
        print("No automatic YAML improvements are available for the current coherence failures.")
        continue

    response = input(
        f"Apply suggested YAML improvements to {process.spec_path.name} for {process.name}? "
        "[y/N or comma-separated indices]: "
    )
    selected_ids = parse_improvement_selection(response, suggestions)
    if not selected_ids:
        print("No YAML changes were applied.")
        continue

    updated_spec, applied_suggestions = apply_process_spec_improvements(spec, suggestions, selected_ids=selected_ids)
    backup_path = write_process_spec_file(process.spec_path, updated_spec)
    print(f"Backup written to: {backup_path}")
    print("Applied YAML improvements:")
    for suggestion in applied_suggestions:
        print(f"- {suggestion['title']}")

    updated_validation_report = validate_process_spec_file(process.spec_path)
    validation_reports[process.name] = updated_validation_report

    if updated_validation_report.get("valid", False):
        updated_spec_loaded = load_process_spec(process.process_dir)
        process_specs[process.name] = updated_spec_loaded
        updated_coherence_report = analyze_process_spec_coherence(updated_spec_loaded)
    else:
        updated_spec_loaded = None
        updated_coherence_report = None

    coherence_reports[process.name] = updated_coherence_report
    display(
        Markdown(
            build_codex_spec_markdown(
                process.name,
                process.spec_path,
                updated_validation_report,
                updated_coherence_report,
                spec=updated_spec_loaded,
            )
        )
    )


## 9. One execution section per discovered process

In [ ]:
summary_rows: list[dict[str, str]] = []
process_results: list[object] = []

for issue in scan.issues:
    display_process_banner(issue.process_name)
    print(f"Discovery error: {issue.message}")
    summary_rows.append(
        {
            "process_name": issue.process_name,
            "status": "discovery_failed",
            "acceptance_passed": "",
            "report_dir": "",
            "generated_files": "",
            "error": issue.message,
        }
    )

for process in selected_processes:
    display_process_banner(process.name)
    print(f"Process directory: {process.process_dir}")
    print(f"Spec path: {process.spec_path}")

    validation_report = validation_reports.get(process.name)
    coherence_report = coherence_reports.get(process.name)

    if not isinstance(validation_report, dict) or not validation_report.get("valid", False):
        print("Skipping execution because structural validation did not pass.")
        summary_rows.append(
            {
                "process_name": process.name,
                "status": "validation_failed",
                "acceptance_passed": "",
                "report_dir": "",
                "generated_files": "",
                "error": first_issue_message({"issues": validation_report.get("errors", []) if isinstance(validation_report, dict) else []}) or "Validation failed",
            }
        )
        continue

    if not isinstance(coherence_report, dict) or not coherence_report.get("passed", False):
        print("Skipping execution because YAML coherence did not pass.")
        summary_rows.append(
            {
                "process_name": process.name,
                "status": "coherence_failed",
                "acceptance_passed": "",
                "report_dir": "",
                "generated_files": "",
                "error": first_issue_message(coherence_report) or "Coherence analysis failed",
            }
        )
        continue

    result = run_process(
        process.process_dir,
        RUNS_ROOT,
        visible=VISIBLE,
        build_mode=BUILD_MODE,
        timeout_seconds=TIMEOUT_SECONDS,
        build_timeout_seconds=BUILD_TIMEOUT_SECONDS,
        report_format=REPORT_FORMAT,
    )

    print(f"Status: {result.status}")
    if result.report_dir:
        print(f"Report directory: {result.report_dir}")
    if result.generated_files:
        print("Generated files:")
        for generated_file in result.generated_files:
            print(f"- {generated_file}")
    if result.error:
        print(f"Error: {result.error}")
    if result.status == "build_failed" and result.details:
        build_diagnostics = result.details.get("build_diagnostics", {})
        session_diagnostics = build_diagnostics.get("diagnostics", {}) if isinstance(build_diagnostics, dict) else {}
        attempts = session_diagnostics.get("import_attempts", [])
        com_builder = session_diagnostics.get("connections_applied", [])
        print(f"Build attempts recorded: {len(attempts)}")
        print(f"Seed files found: {session_diagnostics.get('v14_seed_files_found', [])}")
        print(f"Timed out attempt: {session_diagnostics.get('timed_out_attempt', '')}")
        print(f"Generated INP: {build_diagnostics.get('generated_inp_path')}")
        if isinstance(com_builder, list) and com_builder:
            print(f"COM connections applied: {len(com_builder)}")
            for connection in com_builder[:5]:
                if isinstance(connection, dict):
                    print(f"Connected block: {connection.get('block', '')} inputs={connection.get('inputs', [])} outputs={connection.get('outputs', [])}")
        for warning in session_diagnostics.get("block_parameter_warnings", [])[:5]:
            print(f"Builder warning: {warning}")

    acceptance = result.details.get("acceptance") if result.details else None
    if isinstance(acceptance, dict):
        print(f"Acceptance passed: {acceptance.get('passed')}")

    process_results.append(result)
    summary_rows.append(summarize_process_result(result))


## 10. Codex session analysis per discovered process

In [ ]:
if not process_results:
    print("No process runs available for Codex session analysis.")
else:
    for result in process_results:
        display_process_banner(result.process_name)

        if not result.succeeded:
            print(f"Skipping Codex session analysis because status={result.status!r}.")
            continue

        artifact_paths, artifact_tables = load_result_artifact_tables(result)
        acceptance = result.details.get("acceptance") if result.details else None
        display(
            Markdown(
                build_codex_results_markdown(
                    result.process_name,
                    artifact_paths,
                    artifact_tables,
                    acceptance=acceptance,
                )
            )
        )


## 11. Output summary and validation

In [ ]:
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if summary_df.empty:
    print("No processes were executed.")
else:
    print("Completed process library run.")
    print(summary_df[["process_name", "status", "acceptance_passed", "report_dir"]].to_string(index=False))
